# Attribution Lab — перша транзакція
Порівняння Time Decay, Markov Chain та Shapley Value. Дані синтетичні. Запустіть усі клітинки. A — залучення, B — retention, C — відкриття картки, D — sales. A та C можуть належати одній команді.

Редагуйте CSV: `path` — шлях через `>`; `touch_days` — дні контактів (список у лапках); `end_day` — день завершення; `converted` — 0 або 1; `count` — кількість клієнтів. Після редагування натисніть «Перерахувати».

In [ ]:
%pip -q install ipywidgets plotly
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from itertools import combinations
from math import factorial
import io
import ipywidgets as w
from IPython.display import display, clear_output
import plotly.express as px
seed = pd.DataFrame([
 ['A>B>C>D','0,7,20,30',30,1,100],['A>B>C','0,7,20',30,1,40],
 ['A>C>D','0,15,29',30,1,60],['A>B','0,7',30,0,120],
 ['A>C','0,15',30,0,80],['B>D','0,9',10,1,35],['D','0',1,1,20],
 ['D','0',1,0,60],['A','0',30,0,100],['B','0',30,0,50],
 ['C','0',30,0,40],['','',30,0,100],['','',30,1,10]],
 columns=['path','touch_days','end_day','converted','count'])
def calculate(df,h):
 rows=[]
 for _,r in df.iterrows():
  p=[x.strip() for x in str(r.path).split('>') if x.strip()]; t=[float(x) for x in str(r.touch_days).split(',') if x.strip()]; e=float(r.end_day); y=float(r.converted); n=float(r['count'])
  assert len(p)==len(t) and y in (0,1) and np.isfinite(n) and np.isfinite(e) and e>=0 and n>=0 and all(0<=x<=e for x in t) and t==sorted(t) and not set(p)&{'START','CONV','NULL'}
  rows.append((p,t,e,y,n))
 channels=sorted({c for p,*_ in rows for c in p}); k=len(channels)
 if not k or k>12: raise ValueError('Потрібно від 1 до 12 кампаній.')
 # Time decay: each converting path contributes its count, split across touches.
 td=Counter(); edges=Counter(); groups=Counter(); wins=Counter()
 for p,t,e,y,n in rows:
  s=frozenset(p); groups[s]+=n; wins[s]+=n*y
  if y and p:
   a=np.array(t)-e; w=2**((a-a.max())/h); w=w/w.sum()
   for c,v in zip(p,w): td[c]+=n*v
  chain=['START']+p+['CONV' if y else 'NULL']
  for a,b in zip(chain,chain[1:]): edges[a,b]+=n
 states=['START']+channels; P={}
 for a in states:
  total=sum(v for (x,b),v in edges.items() if x==a)
  P[a]={b:v/total for (x,b),v in edges.items() if x==a and total}
 # Removal redirects incoming probability to NULL (no bypass or renormalization).
 def probability(removed=None):
  active=[a for a in states if a!=removed]; idx={a:i for i,a in enumerate(active)}
  Q=np.zeros((len(active),len(active))); b=np.zeros(len(active))
  for a in active:
   for dest,v in P[a].items():
    if dest=='CONV': b[idx[a]]+=v
    elif dest in idx: Q[idx[a],idx[dest]]+=v
  return float(np.linalg.lstsq(np.eye(len(active))-Q,b,rcond=None)[0][0])
 base=probability(); removal={c:max(0,base-probability(c)) for c in channels}
 # Shapley game: exact observed coalition conversion rate; missing coalitions
 # use pooled rates of observed subsets (explicit demonstration assumption).
 def value(s):
  if groups[s]>0:return wins[s]/groups[s]
  subs=[a for a in groups if a<=s and groups[a]>0]; den=sum(groups[a] for a in subs)
  return sum(wins[a] for a in subs)/den if den else 0.
 sh={}
 for c in channels:
  others=[x for x in channels if x!=c]; score=0.
  for j in range(k):
   for combo in combinations(others,j):
    s=frozenset(combo); score+=factorial(j)*factorial(k-j-1)/factorial(k)*(value(s|{c})-value(s))
  sh[c]=score
 
 def pct(d,c):
  total=sum(d.values()); return 100*d.get(c,0)/total if total>0 else np.nan
 out=pd.DataFrame({'Time Decay %':[pct(td,c) for c in channels],'Markov %':[pct(removal,c) for c in channels],'Shapley (п.п.)':[100*sh[c] for c in channels]},index=channels)
 compare=out[['Time Decay %','Markov %']].copy()
 if sum(sh.values())>0: compare['Shapley %']=[pct(sh,c) for c in channels]
 diagnostics=pd.DataFrame({'Без кампанії':[probability(c) for c in channels],'Падіння, п.п.':[100*removal[c] for c in channels]},index=channels)
 return out,compare,diagnostics,pd.DataFrame(P).T.fillna(0),base,sum(y*n for p,t,e,y,n in rows),sum(y*n for p,t,e,y,n in rows if not p),sum(v>0 for v in groups.values()),2**k,value(frozenset()),value(frozenset(channels))


In [ ]:
slider=w.IntSlider(value=7,min=1,max=90,description='Напіврозпад',continuous_update=False,style={'description_width':'initial'})
editor=w.Textarea(value=seed.to_csv(index=False),layout=w.Layout(width='100%',height='300px'),description='CSV')
run=w.Button(description='Перерахувати',button_style='primary')
reset=w.Button(description='Відновити приклад')
export=w.Button(description='Експорт CSV')
output=w.Output()
result=None

def refresh(*args):
 global result
 result=None
 with output:
  clear_output(wait=True)
  try:
   df=pd.read_csv(io.StringIO(editor.value),keep_default_na=False)
   out,compare,diag,matrix,base,total,organic,observed,possible,v0,vall=calculate(df,slider.value)
   result=compare
   print(f'Транзакції: {total:g} | Без контактів: {organic:g} | Markov P: {base:.2%}')
   display(compare.round(2))
   px.bar(compare.reset_index().melt(id_vars='index',var_name='Модель',value_name='Частка'),x='index',y='Частка',color='Модель',barmode='group',labels={'index':'Кампанія','Частка':'Нормалізований внесок, %'},title='Порівняння моделей').show()
   display(out[['Shapley (п.п.)']].round(3))
   print('Markov: ефект видалення'); display(diag.round(4))
   print('Матриця переходів'); display(matrix.round(3))
   print(f'Shapley: коаліції {observed}/{possible}; базова конверсія {v0:.2%}; повна коаліція {vall:.2%}')
  except Exception as exc:
   print('Помилка даних:',str(exc) or 'Перевірте кількість контактів, дні, converted та count.')

def restore(_):
 editor.value=seed.to_csv(index=False)
 refresh()

def download(_):
 if result is None: return
 result.to_csv('attribution_results.csv',encoding='utf-8-sig')
 from google.colab import files
 files.download('attribution_results.csv')

run.on_click(refresh); reset.on_click(restore); export.on_click(download); slider.observe(refresh,names='value')
display(slider,editor,w.HBox([run,reset,export]),output)
refresh()


## Методологія та обмеження
- Time Decay: вага контакту пропорційна 2^(-давність / напіврозпад); нормалізація в межах конверсійного шляху. Безконтактні транзакції не розподіляються.
- Markov: ланцюг першого порядку; видалені переходи спрямовуються в неконверсію. Частки нормалізують падіння ймовірності конверсії.
- Shapley: середній граничний внесок у конверсію за коаліціями; для відсутніх коаліцій використовується об'єднана конверсія спостережуваних підмножин. Від’ємні внески збережені; нормалізація лише за позитивної суми. Відсотки Шеплі — частки модельного приросту, а не фактичних транзакцій.
- Моделі не доводять причинності. Для production потрібні повні шляхи, порівнянні горизонти спостереження та контрольні експерименти.